# Embedding recommender on quarterly splits

Generate Qwen3-Embedding-0.6B recipe vectors once with `python scripts/embed_recipes.py --device mps --batch-size 16`, then run this notebook. Compare 17 configurations on all 16 validation quarters, with 10,000 candidates. Descriptions use the first 64 words. Test quarters are not evaluated; no final model is selected.

In [ ]:
import sys
sys.path.insert(0, "../src")

import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from remy.paths import PROJECT_ROOT
from remy.splits import RollingSplits
from remy.evaluate import evaluate_quarter
from remy.model.embedding import EmbeddingRecommender, load_embeddings, EMBEDDING_DIR, TEXT_FEATURES
from remy.model.popularity import Popularity

In [ ]:
baseline = dict(
    rating_weights={5: 1., 4: .5, 3: 0., 2: 0., 1: 0., 0: 0.},
    feature_weights=dict(name=1., ingredients=1., tags=1., description=1., numeric=.5),
    n_candidates=10000,
)
experiments = {
    "baseline": baseline,
    "five_star_only": {**baseline, "rating_weights": {**baseline["rating_weights"], 4: 0.}},
    "equal_positive": {**baseline, "rating_weights": {**baseline["rating_weights"], 4: 1.}},
    "no_numeric": {**baseline, "feature_weights": {**baseline["feature_weights"], "numeric": 0.}},
    "no_description": {**baseline, "feature_weights": {**baseline["feature_weights"], "description": 0.}},
    "ingredient_emphasis": {**baseline, "feature_weights": {**baseline["feature_weights"], "ingredients": 2.}},
}
features = TEXT_FEATURES + ["numeric"]
for feature in features:
    experiments[f"{feature}_only"] = {**baseline, "feature_weights": {f: float(f == feature) for f in features}}
for name, numeric_weight in [("ingredients_tags", 0.), ("ingredients_tags_numeric", .5)]:
    experiments[name] = {**baseline, "feature_weights": dict(name=0., ingredients=1., tags=1., description=0., numeric=numeric_weight)}
for name in ["baseline", "ingredients_only", "ingredients_tags", "ingredients_tags_numeric"]:
    experiments[f"{name}_top3"] = {**experiments[name], "aggregation": "top_history", "history_k": 3}
assert len(experiments) == 17
metrics = ["NDCG", "HR@10", "HR@100", "Recall@10", "Recall@100", "candidate_recall"]
list(experiments)


## Sanity checks

Small synthetic checks, including missing features and users without positive ratings.

In [ ]:
import numpy as np
import pandas as pd
from remy.model.embedding import EmbeddingRecommender, TEXT_FEATURES, FEATURE_WEIGHTS
from remy.model.popularity import Popularity
from remy.splits import Quarter
from remy.evaluate import evaluate_quarter

rng = np.random.default_rng(10718)
e = {'id': np.arange(1, 7), 'numeric': rng.uniform(1, 10, (6, 10)).astype('float32')}
e['numeric'][:, 0] = 1
for f in TEXT_FEATURES:
    v = rng.normal(size=(6, 4))
    e[f] = (v / np.linalg.norm(v, axis=1, keepdims=True)).astype('float16')
e['tags'][1] = 0
e['description'][4] = 0
h = pd.DataFrame({'user_id': [1,1,1,2,2,2], 'recipe_id': [1,2,3,1,2,4], 'rating': [5,4,1,1,0,2]})
r = pd.DataFrame({'id': np.arange(1, 7)})
for aggregation in ['centroid', 'top_history']:
    m = EmbeddingRecommender(e, aggregation=aggregation).fit(h, r)
    assert m.ranking_ == Popularity().fit(h, r).ranking_
    assert m.recommend_many([2, 999]) == {}
    rec = m.recommend_many([1])
    assert set(rec[1]) == {4, 5, 6}
    assert np.isfinite(m._scores(0)).all()
    expected = e['ingredients'][0].astype('float32') + .5 * e['ingredients'][1].astype('float32')
    expected /= np.linalg.norm(expected)
    assert np.allclose(m.user_vectors_['ingredients'][0], expected, atol=1e-6)
    q = Quarter(pd.Timestamp('2007-01-01'), pd.Timestamp('2007-04-01'), h, r,
                pd.DataFrame({'user_id': [1,2]}), pd.DataFrame({'user_id': [1,2], 'recipe_id': [5,6], 'rating': [5,5]}))
    assert np.isfinite(evaluate_quarter(q, m.ranking_, m.seen_, rec)['NDCG'])
    weights = {f: float(f == 'ingredients') for f in FEATURE_WEIGHTS}
    ingredient_model = EmbeddingRecommender(e, aggregation=aggregation, feature_weights=weights).fit(h, r)
    if aggregation == 'top_history':
        a = m.normalize(e['ingredients'][[0,1]].astype('float32'))
        b = ingredient_model.candidate_vectors_['ingredients']
        expected = ((b @ a.T) * [1., .5]).sum(axis=1) / 1.5
        assert np.allclose(ingredient_model._scores(0), expected, atol=1e-6)
# With no usable description signal, use the evaluator's popularity fallback.
e['description'][:] = 0
weights = {f: float(f == 'description') for f in FEATURE_WEIGHTS}
for aggregation in ['centroid', 'top_history']:
    m = EmbeddingRecommender(e, aggregation=aggregation, feature_weights=weights).fit(h, r)
    assert m.recommend_many([1]) == {1: []}
print('Sanity checks passed: weighted profiles, top-history formula, missing features, fallback, seen exclusion, constant numeric columns, evaluator integration.')


In [ ]:
splits = RollingSplits()
quarters = splits.evaluation_quarters("val")
assert len(quarters) == 16
assert all((q.history.date < q.start).all() and (q.recipes.submitted < q.start).all() for q in quarters)
embeddings = load_embeddings()
assert np.array_equal(np.sort(splits.recipes.id), embeddings["id"])
output = PROJECT_ROOT / "data/processed/embedding_experiments"
output.mkdir(parents=True, exist_ok=True)
metadata = json.loads((EMBEDDING_DIR / "metadata.json").read_text())
settings = json.loads(json.dumps(dict(encoder_revision=metadata["revision"], description_words=metadata["description_words"], experiments=experiments)))
plan = output / "settings.json"
if plan.exists():
    assert json.loads(plan.read_text()) == settings, "Settings changed; use a fresh output directory."
else:
    plan.write_text(json.dumps(settings, indent=2))


## Fit and evaluate

The first six settings reproduce the requested baseline ablations. Single-feature and ingredient/tag combinations test which content signals help. Four top-three-history variants test matching individual liked recipes instead of one averaged taste profile. Numerical similarity still uses averaged numerical preferences.

Each setting uses a fresh model at each cutoff. Checkpoints are saved after each quarter. Every configuration evaluates all validation quarters and all scored users.

In [ ]:
all_results = {}
total_start = time.perf_counter()
for name, model_kwargs in experiments.items():
    path = output / f"qwen_{name}_val.csv"
    rows = pd.read_csv(path, parse_dates=["start"]).to_dict("records") if path.exists() else []
    assert len(rows) <= len(quarters)
    assert all(row["start"] == quarters[i].start for i, row in enumerate(rows))
    for quarter in quarters[len(rows):]:
        start = time.perf_counter()
        model = EmbeddingRecommender(embeddings, **model_kwargs).fit(quarter.history, recipes=quarter.recipes)
        users = quarter.outcomes.loc[quarter.outcomes.rating == 5, "user_id"].unique()
        recs = model.recommend_many(users)
        assert set(model.candidates_).issubset(set(quarter.recipes.id))
        assert all(not(set(v) & model.seen_.get(u, set())) for u, v in recs.items())
        row = evaluate_quarter(quarter, model.ranking_, model.seen_, recs)
        row["seconds"] = time.perf_counter() - start
        rows.append(row)
        temporary = path.with_suffix(".partial.csv")
        pd.DataFrame(rows).to_csv(temporary, index=False)
        temporary.replace(path)
        print(f"{name}: {quarter.start.date()}, {row['seconds']/60:.2f} min, NDCG={row['NDCG']:.5f}")
        del model, recs
    all_results[name] = pd.DataFrame(rows)
    print(f"Finished {name}: {sum(row['seconds'] for row in rows)/60:.1f} min across 16 quarters")
print(f"Total elapsed: {(time.perf_counter()-total_start)/3600:.2f} hours")


## Compare validation results

Popularity is evaluated on the same quarters. Results are displayed together; no configuration is selected and no test evaluation runs.

In [ ]:
rows = []
for quarter in quarters:
    model = Popularity().fit(quarter.history, recipes=quarter.recipes)
    rows.append(evaluate_quarter(quarter, model.ranking_, model.seen_))
popularity_results = pd.DataFrame(rows)
popularity_results.to_csv(output / "popularity_val.csv", index=False)
summary = pd.DataFrame({name: table[metrics].mean() for name, table in all_results.items()}).T
summary.loc["popularity"] = popularity_results.reindex(columns=metrics).mean()
summary["minutes"] = pd.Series({name: table.seconds.sum()/60 for name, table in all_results.items()})
summary = summary.sort_values("NDCG", ascending=False)
summary.to_csv(output / "validation_summary.csv")
summary.round(5)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 7), sharey=True)
summary.sort_values("NDCG").plot.barh(y="NDCG", ax=axes[0], legend=False)
summary.loc[summary.sort_values("NDCG").index].plot.barh(y="Recall@10", ax=axes[1], legend=False)
axes[0].set(xlabel="Mean NDCG", ylabel="Configuration")
axes[1].set(xlabel="Mean Recall@10", ylabel="")
fig.tight_layout()
plt.show()


## Optional ranking diagnostics

Inspect component ranks of in-pool positives for 30 reproducibly sampled users in the first validation quarter. This sample explains behavior; the results above use all validation users.

In [ ]:
quarter = quarters[0]
model = EmbeddingRecommender(embeddings).fit(quarter.history, recipes=quarter.recipes)
positives = quarter.outcomes[quarter.outcomes.rating == 5]
ids = positives.user_id.unique()
users = np.random.default_rng(10718).choice(ids, min(30, len(ids)), replace=False)
rows = []
for user in users:
    if user not in model.user_row_.index:
        continue
    relevant = set(positives.loc[positives.user_id == user, "recipe_id"])
    unseen = np.array([r not in model.seen_[user] for r in model.candidates_])
    popularity_rank = {r: i+1 for i, r in enumerate(model.candidates_[unseen])}
    for feature in features + ["combined"]:
        model.feature_weights = baseline["feature_weights"] if feature == "combined" else {f: float(f == feature) for f in features}
        scores = model._scores(model.user_row_.loc[user])
        order = np.lexsort((model.candidates_, -scores))
        ranked = model.candidates_[order[unseen[order] & np.isfinite(scores[order])]]
        rank = {r: i+1 for i, r in enumerate(ranked)}
        rows.extend(dict(user_id=user, recipe_id=r, feature=feature, rank=rank[r], popularity_rank=popularity_rank[r])
                    for r in relevant & set(rank))
del model
diagnostics = pd.DataFrame(rows)
if not diagnostics.empty:
    display(diagnostics.groupby("feature")[["rank", "popularity_rank"]].median())
